<a href="https://colab.research.google.com/github/sunnysavita10/Generative-AI-Indepth-Basic-to-Advance/blob/main/MergerRetriever_and_LongContextReorder.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Install the Require Libraries

In [ ]:
!pip install -qU langchain chromadb huggingface_hub sentence-transformers pypdf openai tiktoken

In [ ]:
!pip install -U langchain-community

# Let's Load the Data Now...

In [ ]:
from langchain.document_loaders import PyPDFLoader

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
loader_harrypotter  = PyPDFLoader("/content/harry_potter_book.pdf")

In [ ]:
documnet_harrypotter = loader_harrypotter.load()

In [ ]:
print(len(documnet_harrypotter))

In [ ]:
loader_got = PyPDFLoader("/content/got_book.pdf")

In [ ]:
documnet_got = loader_got.load()

In [ ]:
print(len(documnet_got))


# Let's create the text splitter for chunking

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

In [ ]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=100)

In [ ]:
text_harrypotter = text_splitter.split_documents(documnet_harrypotter)

In [ ]:
text_got = text_splitter.split_documents(documnet_got)

In [ ]:
print(len(text_harrypotter))

In [ ]:
print(len(text_got))

# Load the Embedding Model to Conver the Data into Vector

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings, OpenAIEmbeddings,HuggingFaceBgeEmbeddings

In [ ]:
HF_TOKEN_REMOVEDembeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

In [ ]:
HF_TOKEN_REMOVEDbge_embeddings = HuggingFaceBgeEmbeddings(model_name="BAAI/bge-large-en")


In [ ]:
from google.colab import userdata
OPENAI_API_KEY=userdata.get('OPENAI_API_KEY')

In [ ]:
os.environ["OPENAI_API_KEY"]=OPENAI_API_KEY

In [ ]:
openai_embeddings = OpenAIEmbeddings(openai_api_key=OPENAI_API_KEY)

# Now ingest the Data into the Chroma Database

In [ ]:
from langchain.vectorstores import Chroma
import chromadb

In [ ]:
import os
os.getcwd()

In [ ]:
CURRENT_DIR = os.path.dirname(os.path.abspath("."))

In [ ]:
CURRENT_DIR

In [ ]:
DB_DIR = os.path.join(CURRENT_DIR, "/content/db")

In [ ]:
DB_DIR

In [ ]:
client_settings = chromadb.config.Settings(
    is_persistent=True,
    persist_directory=DB_DIR,
    anonymized_telemetry=False,
)

In [ ]:
harrypotter_vectorstore = Chroma.from_documents(text_harrypotter,
                                       HF_TOKEN_REMOVEDbge_embeddings,
                                       client_settings=client_settings,
                                       collection_name="harrypotter",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/harrypotter")

In [ ]:
got_vectorstore = Chroma.from_documents(text_got,
                                       HF_TOKEN_REMOVEDbge_embeddings,
                                       client_settings=client_settings,
                                       collection_name="got",
                                       collection_metadata={"hnsw":"cosine"},
                                       persist_directory="/store/got")

 # Now Crearte a Retriever

In [ ]:
retriever_harrypotter = harrypotter_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

In [ ]:
retriever_got = got_vectorstore.as_retriever(search_type="mmr",search_kwargs={"k": 5, "include_metadata": True})

# Let's Merge both Retriever

# It is also called lord of retriever(LOTR)

In [ ]:
from langchain.retrievers.merger_retriever import MergerRetriever

In [ ]:
lotr = MergerRetriever(retrievers=[retriever_harrypotter, retriever_got])

In [ ]:
for chunks in lotr.get_relevant_documents("Who was the jon snow?"):
    print(chunks.page_content)

In [ ]:
for chunks in lotr.get_relevant_documents("Who is a harry potter?"):
    print(chunks.page_content)

## See this result is too much messy now lets refine it according to the question and overcome the situation of lost in middle

# Now After understanding step by step it create a pipeline for LLM

In [ ]:
from langchain.document_transformers import (
    EmbeddingsClusteringFilter,
    EmbeddingsRedundantFilter,
)
from langchain.retrievers.document_compressors import DocumentCompressorPipeline
from langchain.retrievers import ContextualCompressionRetriever
from langchain.document_transformers import LongContextReorder

In [ ]:
from re import search
filter = EmbeddingsRedundantFilter(embeddings=HF_TOKEN_REMOVEDbge_embeddings)
reordering = LongContextReorder()
pipeline = DocumentCompressorPipeline(transformers=[filter, reordering])
compression_retriever_reordered = ContextualCompressionRetriever(
    base_compressor=pipeline, base_retriever=lotr,search_kwargs={"k": 3, "include_metadata": True}
)


In [ ]:
"""docs = compression_retriever_reordered.get_relevant_documents("What is esops?")
print(len(docs))
#
print(docs[0].page_content)"""

# Load the model from huggingface

In [ ]:
!pip install llama-cpp-python

In [ ]:
from langchain.llms import LlamaCpp
llms = LlamaCpp(streaming=True,
                   model_path="/content/drive/MyDrive/zephyr-7b-beta.Q4_K_M.gguf",
                   max_tokens = 1500,
                   temperature=0.75,
                   top_p=1,
                   gpu_layers=0,
                   stream=True,
                   verbose=True,n_threads = int(os.cpu_count()/2),
                   n_ctx=4096)

In [ ]:
from langchain.chains import RetrievalQA

In [ ]:
qa = RetrievalQA.from_chain_type(
      llm=llms,
      chain_type="stuff",
      retriever = compression_retriever_reordered,
      return_source_documents = True
)

In [ ]:
query ="who is jon snow?"
results = qa(query)
print(results['result'])
#
print(results["source_documents"])


 Jon Snow is a character in George R.R. Martin's "A Song of Ice and Fire" series, which has been adapted into the popular HBO show "Game of Thrones." In the context provided, he is described as Bran Stark's bastard brother and is mentioned as moving closer to Bran during a scene in which Bran witnesses his father, King Robert Baratheon, sentencing Ned Stark to death for treason. Jon Snow has been portrayed by actor Kit Harington on the TV show "Game of Thrones."

In [ ]:
results = qa("who is a harry potter?")
print(results['result'])
#
print(results["source_documents"])
#
for source in  results["source_documents"]:
    print(source.metadata)

Harry Potter is the main character in J.K. Rowling's series of novels and films about magic and wizards. He is an orphan who discovers that he has magical powers and goes to attend Hogwarts School of Witchcraft and Wizardry, where he makes friends and battles evil forces like Lord Voldemort.

In [ ]:
results = qa.invoke("How does Jon Snow's relationship with the Stark family influence his identity and decisions throughout A Game of Thrones?")


In [ ]:
results

In [ ]:
print(results['result'])



In [ ]:
print(results["source_documents"])



In [ ]:
for source in  results["source_documents"]:
    print(source.metadata)